In [1]:
# 필요한 라이브러리
import streamlit as st
import pandas as pd
import matplotlib.pyplot as plt

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

import time

In [2]:
def get_jobkorea_data():
    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service)
    wait = WebDriverWait(driver, 10)

    driver.get('https://www.jobkorea.co.kr/')
    try:
        popup_close_btn = wait.until(EC.element_to_be_clickable((By.ID, 'layerSystemCheck')))
        popup_close_btn.click()
    except:
        pass

    search_box = wait.until(EC.presence_of_element_located((By.ID, 'stext')))
    search_box.send_keys("데이터분석")

    search_button = wait.until(EC.element_to_be_clickable((By.ID, 'common_search_btn')))
    search_button.click()

    time.sleep(2)

    soup = BeautifulSoup(driver.page_source, 'html.parser')
    target_section = soup.select_one('section.content-recruit.on')
    job_cards = target_section.select('article.list-item')

    result_list = []
    for job in job_cards:
        try:
            site = "Job_Korea"
            company_tag = job.select_one('.list-section-corp a[target]')
            company = company_tag.text.strip() if company_tag else None

            title_tag = job.select_one('.list-section-information .information-title a')
            title = title_tag.text.strip() if title_tag else None

            detail_tags = job.select('.chip-information-group li')
            details = ' / '.join([li.text.strip() for li in detail_tags]) if detail_tags else None

            url_tail = job.get('data-gavirturl')
            full_url = f"https:{url_tail}" if url_tail else None

            result_list.append({
                'Site': site,
                'Col_Company': company,
                'Col_Recruit': title,
                'Col_detail': details,
                'Col_url': full_url
            })
        except:
            continue

    driver.quit()
    return pd.DataFrame(result_list)

In [3]:
jobkorea_df = get_jobkorea_data()
jobkorea_df.to_csv('data_jobkorea.csv', index=False, encoding='utf-8-sig')